<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

# 🎙️ Chatterbox TTS Fine-Tuning & LoRA Inference Kit

This Colab notebook provides a complete environment for fine-tuning Chatterbox TTS models with your custom dataset using either **LoRA** (efficient, recommended for <10h data) or **Full Fine-Tuning**.

## Features:
- ✅ GPU-accelerated training with CUDA
- ✅ LoRA support for efficient training (~60% less VRAM)
- ✅ Turbo and Standard model support
- ✅ Automatic preprocessing and model preparation
- ✅ Built-in inference and model merging

---

## Step 1: Clone Repository & Install Dependencies

In [ ]:
#@title 📦 Clone Repository and Install Dependencies
#@markdown Run this cell to clone the repository and install all required packages.

import os

# Clone the repository if not already present
if not os.path.exists("chatterbox-finetuning"):
    !git clone https://github.com/amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
else:
    %cd chatterbox-finetuning

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install Python dependencies
!pip install chatterbox-tts --no-deps
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 torchcodec --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
!pip install peft==0.17.1  --no-deps
!pip install -q -r requirements.txt



## Step 2: Configure Training Settings

In [ ]:
#@title ⚙️ Configuration Settings
#@markdown Adjust these settings based on your needs before running setup.

#@markdown ### Model Selection
is_turbo = True  #@param {type:"boolean"}
#@markdown - **Turbo Mode** (`True`): GPT-2 based, faster training, BPE tokenizer
#@markdown - **Standard Mode** (`False`): Llama-based, grapheme tokenizer

#@markdown ### Training Strategy
is_lora = True  #@param {type:"boolean"}
#@markdown - **LoRA** (`True`): Recommended for datasets <10 hours, uses ~60% less VRAM
#@markdown - **Full Fine-Tune** (`False`): For massive datasets >10 hours

is_merge_lora = True  #@param {type:"boolean"}
#@markdown - **Auto-merge LoRA** (`True`): Automatically merge LoRA weights after training
#@markdown - **Manual Merge** (`False`): Keep LoRA adapters separate

#@markdown ### Dataset Settings
project_name = "<project_name>"  #@param {type:"string"}
#@markdown Your project name (used for organizing files)

base_output_dir = "./chatterbox_output"  #@param {type:"string"}
#@markdown Base output directory for training outputs

base_dataset_dir = "./MyTTSDataset"  #@param {type:"string"}
#@markdown Base directory containing your dataset files

ljspeech_format = True  #@param {type:"boolean"}
#@markdown Set to `True` if your dataset is in LJSpeech format (metadata.csv)

json_format = False  #@param {type:"boolean"}
#@markdown Set to `True` if your dataset is in JSON format

preprocess = "auto"  #@param ["True", "False", "auto"]
#@markdown - **True**: Always run preprocessing
#@markdown - **False**: Skip preprocessing
#@markdown - **auto**: Smart detection (run if needed)

#@markdown ### LoRA Configuration
lora_r = 128  #@param {type:"integer"}
#@markdown LoRA rank (dimension of low-rank matrices)

lora_alpha = 256  #@param {type:"integer"}
#@markdown LoRA alpha (scaling factor for LoRA weights)

#@markdown ### Hyperparameters
batch_size = 8  #@param {type:"integer"}
#@markdown Batch size per step (adjust based on VRAM: 2, 4, 8)

grad_accum = 4  #@param {type:"integer"}
#@markdown Gradient accumulation steps (Effective Batch Size = Batch * Accum)

learning_rate = 0.0001  #@param {type:"number"}
#@markdown Learning rate (T3 is sensitive, keep low: 1e-4 for LoRA, 1e-5 for full fine-tune)

num_epochs = 10  #@param {type:"integer"}
#@markdown Number of training epochs (10 for LoRA, 30 for full fine-tune)

save_steps = 500  #@param {type:"integer"}
#@markdown Save checkpoint every N steps

save_total_limit = 5  #@param {type:"integer"}
#@markdown Maximum number of checkpoints to keep

dataloader_num_workers = 8  #@param {type:"integer"}
#@markdown Number of workers for data loading

#@markdown ### Constraints
max_text_len = 256  #@param {type:"integer"}
#@markdown Maximum text token length

max_speech_len = 850  #@param {type:"integer"}
#@markdown Maximum speech frame length (truncates very long audio)

prompt_duration = 3.0  #@param {type:"number"}
#@markdown Duration for the reference prompt (seconds)

#@markdown ### Inference Settings
inference_prompt_path = "./speaker_reference/2.wav"  #@param {type:"string"}
#@markdown Path to reference audio for inference

inference_test_text = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."  #@param {type:"string"}
#@markdown Test text for inference

import os
import re

# Read the current config.py to preserve structure
config_file_path = 'src/config.py'
with open(config_file_path, 'r') as f:
    original_config = f.read()

# Generate new TrainConfig class content with all parameters
new_train_config = f"""class TrainConfig:
    """
    """\n"""
    # Auto-generated configuration from Colab - Step 2
    """
    """\n"""
    is_turbo: bool = {str(is_turbo).lower()}
    """
    """\n"""
    is_lora: bool = {str(is_lora).lower()}
    """
    """\n"""
    is_merge_lora: bool = {str(is_merge_lora).lower()}
    """
    """\n"""
    project_name: str = \"{project_name}\"
    """
    """\n"""
    base_output_dir: str = \"{base_output_dir}\"
    """
    """\n"""
    base_dataset_dir: str = \"{base_dataset_dir}\"
    """
    """
    ljspeech = {str(ljspeech_format).lower()} # Set True if the dataset format is ljspeech, and False if it's file-based.
    """
    """\n"""
    json_format = {str(json_format).lower()}
    """
    """\n"""
    preprocess: Optional[Literal[True, False, \"auto\"]] = {repr(preprocess)}
    """
    """\n"""
    lora_r: int = {lora_r}
    """
    """\n"""
    lora_alpha: int = {lora_alpha}
    """
    """\n"""
    batch_size: int = {batch_size}
    """
    """\n"""
    grad_accum: int = {grad_accum}
    """
    """\n"""
    learning_rate: float = {learning_rate}
    """
    """\n"""
    num_epochs: int = {num_epochs}
    """
    """\n"""
    save_steps: int = {save_steps}
    """
    """\n"""
    save_total_limit: int = {save_total_limit}
    """
    """\n"""
    dataloader_num_workers: int = {dataloader_num_workers}
    """
    """\n"""
    max_text_len: int = {max_text_len}
    """
    """\n"""
    max_speech_len: int = {max_speech_len}
    """
    """\n"""
    prompt_duration: float = {prompt_duration}
    """
    """\n"""
    inference_prompt_path: str = \"{inference_prompt_path}\"
    """
    """\n"""
    inference_test_text: str = \"{inference_test_text}\"
"""

# Replace the TrainConfig class in the original config
pattern = r'class TrainConfig:.*?(?=\nclass |\Z)'
updated_config = re.sub(pattern, new_train_config.strip(), original_config, flags=re.DOTALL)

# Write the updated config back
with open(config_file_path, 'w') as f:
    f.write(updated_config)

print(f"✅ Configuration written to src/config.py:")
print(f"  - Turbo Mode: {is_turbo}")
print(f"  - LoRA Training: {is_lora}")
print(f"  - Auto-merge LoRA: {is_merge_lora}")
print(f"  - Project Name: {project_name}")
print(f"  - Base Output Dir: {base_output_dir}")\nprint(f"  - Base Dataset Dir: {base_dataset_dir}")\nprint(f"  - Dataset Format: {'LJSpeech' if ljspeech_format else ('JSON' if json_format else 'File-Based')}")
print(f"  - Preprocessing: {preprocess}")
print(f"  - LoRA Rank: {lora_r}, Alpha: {lora_alpha}")
print(f"  - Batch Size: {batch_size}, Grad Accum: {grad_accum}")
print(f"  - Learning Rate: {learning_rate}")
print(f"  - Epochs: {num_epochs}")
print(f"  - Max Text Len: {max_text_len}, Max Speech Len: {max_speech_len}")
print(f"  - Prompt Duration: {prompt_duration}s")
print(f"  - Save Steps: {save_steps}, Save Limit: {save_total_limit}")


In [ ]:
%cd chatterbox-finetuning/

/content/chatterbox-finetuning


## Step 3: Download and Prepare Base Models

In [ ]:
#@title 📥 Download Base Models and Prepare Tokenizer
#@markdown This downloads the necessary base models and prepares the tokenizer. **Must run before training!**

# Clean pretrained_models if switching modes
if os.path.exists("pretrained_models"):
    import shutil
    shutil.rmtree("pretrained_models")
    print("🗑️ Cleaned old pretrained_models directory")

# Run setup.py
print("\n⏳ Downloading and preparing models...\n")
!python setup.py

print("\n✅ Setup complete! Check the output above for the new_vocab_size value.")

## Step 4: Start Training

In [ ]:
#@title 🏋️ Start Training
#@markdown This will preprocess your dataset and start training. The process includes:
#@markdown 1. Preprocessing (audio feature extraction)
#@markdown 2. Model initialization with extended vocabulary
#@markdown 3. Training with periodic inference checks
#@markdown
#@markdown **Expected time:** 30 mins - 2 hours depending on dataset size and GPU

print("Starting training...")
print(f"Mode: {'Turbo' if is_turbo else 'Standard'} + {'LoRA' if is_lora else 'Full Fine-Tune'}")
print(f"Project: {project_name}")
print("\nMonitoring training progress...\n")

# Run training
!python train.py

print("\n✅ Training complete!")
print("\nOutput location:")
if is_lora:
    print(f"  LoRA adapter: chatterbox_output/{project_name}/new_lang_adapter/")
else:
    model_name = "t3_turbo_finetuned.safetensors" if is_turbo else "t3_finetuned.safetensors"
    print(f"  Full model: chatterbox_output/{project_name}/{model_name}")

##Test Inference

In [ ]:
#@title 🗣️ Test Speech Synthesis (Inference)
#@markdown Generate speech using your trained model. Make sure you have a reference audio file.

import os
from IPython.display import Audio, display

# Check for speaker reference
ref_dir = "speaker_reference"
if not os.path.exists(ref_dir):
    os.makedirs(ref_dir)
    print(f"Created {ref_dir}/ directory")
    print("\n⚠️ Please upload a reference audio file (3-10 seconds clean speech)")
    print(f"   Upload it to: {ref_dir}/reference.wav")
else:
    ref_files = [f for f in os.listdir(ref_dir) if f.endswith('.wav')]
    if ref_files:
        print(f"Found reference files: {ref_files}")

        # Update inference.py with test text
        test_text = "Hello, this is a test of my custom voice model."  #@param {type:"string"}
        ref_file = ref_files[0]  #@param {type:"string"}

        # Modify inference.py temporarily
        with open('inference.py', 'r') as f:
            inf_content = f.read()

        # Replace test text
        inf_content = inf_content.replace(
            'TEXT_TO_SAY = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."',
            f'TEXT_TO_SAY = "{test_text}"'
        )

        with open('inference.py', 'w') as f:
            f.write(inf_content)

        print(f"\nRunning inference with:")
        print(f"  Text: {test_text}")
        print(f"  Reference: {ref_dir}/{ref_file}")
        print("\nGenerating speech...\n")

        # Run inference
        !python inference.py

        # Play output
        if os.path.exists("output.wav"):
            print("\n✅ Generated output.wav")
            display(Audio("output.wav"))
        else:
            print("\n❌ Output file not generated. Check error messages above.")
    else:
        print(f"\n⚠️ No .wav files found in {ref_dir}/")
        print("Please upload a reference audio file (3-10 seconds of clean speech)")

##Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you used LoRA training and are satisfied with the results, merge the adapter into a standalone model file.

if is_lora:
    print("Merging LoRA adapter into base model...")
    print("This creates a single .safetensors file ready for deployment.\n")

    !python merge_lora.py

    print("\n✅ Merge complete!")
    merged_model = f"chatterbox_output/{project_name}/t3_turbo_finetuned_merged.safetensors" if is_turbo else f"chatterbox_output/{project_name}/t3_finetuned_merged.safetensors"
    print(f"Merged model saved to: {merged_model}")

    if os.path.exists(merged_model):
        size_mb = os.path.getsize(merged_model) / (1024 * 1024)
        print(f"File size: {size_mb:.1f} MB")
else:
    print("ℹ️ Skipping merge - you used full fine-tuning, not LoRA")

##Download Trained Model

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your trained model to Google Drive or local storage.

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Determine what to save
output_dir = f"chatterbox_output/{project_name}"

if os.path.exists(output_dir):
    # Create backup directory in Drive
    drive_backup = f"/content/drive/MyDrive/chatterbox_models/{project_name}"
    os.makedirs(drive_backup, exist_ok=True)

    # Copy all output files
    for item in os.listdir(output_dir):
        src = os.path.join(output_dir, item)
        dst = os.path.join(drive_backup, item)

        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {item} ({os.path.getsize(src) / (1024*1024):.1f} MB)")
        elif os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ Copied directory: {item}/")

    print(f"\n📁 All files saved to: {drive_backup}")
    print("\nYou can also download individual files using the file browser on the left.")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training has completed successfully.")

---
## Troubleshooting Tips

### Common Issues:

1. **CUDA Out of Memory**
   - Enable LoRA training (`is_lora = True`)
   - Reduce `batch_size` in `src/config.py`
   - Use a smaller dataset

2. **Poor Quality Output**
   - Ensure reference audio is clean (3-10 seconds)
   - Check dataset quality (minimal background noise)
   - Train for more epochs if dataset is small
   - For Turbo model: switch to LoRA if experiencing instability

3. **Vocabulary Mismatch Error**
   - Make sure `new_vocab_size` matches the tokenizer
   - Re-run `setup.py` after changing modes

4. **Missing Reference Audio**
   - Upload a clean 3-10 second WAV file to `speaker_reference/`
   - Use the same speaker as your training dataset for best results

---
**Need Help?** Check the README.md file or open an issue on GitHub.